In [1]:
import requests
import os
import time
import zipfile
import pandas as pd
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)
bucket = "crypto-data-lake"
if not client.bucket_exists(bucket):
    client.make_bucket(bucket)

In [3]:
spark = SparkSession.builder \
    .appName("LandingZone") \
    .config("spark.master", "spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.jars", ",".join([
        "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
        "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
    ])) \
    .getOrCreate()

25/10/03 15:49:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [8]:
def download_file(url, file_name):
    if os.path.exists(file_name):
        print(f"{file_name} exits")
        return
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(file_name, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        print(f"Downloaded {file_name} {(os.path.getsize(file_name) / (1024 * 1024)):.2f}MB completed")

def remove_file(file_name):
    if os.path.exists(file_name):
        os.remove(file_name)
        print(f"{file_name} removed")
    else:
        print(f"{file_name} not found")

def extract_file(extract_dir, zip_path):
    if not os.path.exists(extract_dir):
        os.makedirs(extract_dir)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

In [9]:
extract_dir = "unzipped_data"

In [10]:
urls = [f"https://data.binance.vision/data/spot/daily/aggTrades/BTCUSDT/BTCUSDT-aggTrades-2025-08-{i:02d}.zip" for i in range(1, 2)]

In [12]:
file_names = [u.split("/")[-1] for u in urls]

In [15]:
for url, file_name in zip(urls, file_names):
    start_t = time.time()
    download_file(url, file_name)
    # remove_file(file_name)
    end_t = time.time()
    print(f"Processed in {(end_t - start_t):.3f} seconds")

Downloaded BTCUSDT-aggTrades-2025-08-01.zip 18.64MB completed
Processed in 3.355 seconds


In [16]:
!ls -lh unzipped_data/

total 110M
-rw-r--r-- 1 jovyan jovyan 110M Sep 21 14:33 BTCUSDT-aggTrades-2025-08-01.csv


In [17]:
remove_file("unzipped_data/BTCUSDT-aggTrades-2025-08-01.csv")

unzipped_data/BTCUSDT-aggTrades-2025-08-01.csv removed


In [19]:
zip_path = "BTCUSDT-aggTrades-2025-08-01.zip"
extract_file(extract_dir, zip_path)

In [20]:
csv_file = os.path.join(extract_dir, os.listdir(extract_dir)[0])
print(f"Extracted CSV: {csv_file}")

Extracted CSV: unzipped_data/BTCUSDT-aggTrades-2025-08-01.csv


In [21]:
schema = types.StructType([
    types.StructField('agg_trade_id', types.LongType(), True), 
    types.StructField('price', types.DoubleType(), True), 
    types.StructField('quantity', types.DoubleType(), True), 
    types.StructField('first_trade_id', types.LongType(), True), 
    types.StructField('last_trade_id', types.LongType(), True), 
    types.StructField('timestamp', types.LongType(), True), 
    types.StructField('is_buyer_maker', types.BooleanType(), True), 
    types.StructField('is_best_match', types.BooleanType(), True)
])

In [22]:
df = spark.read \
    .option("header", "false") \
    .schema(schema) \
    .csv(csv_file)

In [23]:
df.printSchema()

root
 |-- agg_trade_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- first_trade_id: long (nullable = true)
 |-- last_trade_id: long (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)



In [24]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+------------+-----+--------+--------------+-------------+---------+--------------+-------------+
|agg_trade_id|price|quantity|first_trade_id|last_trade_id|timestamp|is_buyer_maker|is_best_match|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+
|           0|    0|       0|             0|            0|        0|             0|            0|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+



In [25]:
df.describe().show()

25/10/03 02:27:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 3:>                                                          (0 + 2) / 2]

+-------+-----------------+------------------+--------------------+-------------------+-------------------+--------------------+
|summary|     agg_trade_id|             price|            quantity|     first_trade_id|      last_trade_id|           timestamp|
+-------+-----------------+------------------+--------------------+-------------------+-------------------+--------------------+
|  count|          1314072|           1314072|             1314072|            1314072|            1314072|             1314072|
|   mean|   3.6411511565E9|114772.51094437456|0.018634520832961927|5.124889106730406E9|5.124889108896769E9|1.754049467280027...|
| stddev|379340.0558048148| 815.0145912458705| 0.13296616458618138| 1227424.3654440136| 1227424.8011467636|2.531215074523887...|
|    min|       3640494121|         112722.58|              1.0E-5|         5122977554|         5122977554|    1754006400328945|
|    max|       3641808192|          116052.0|             26.5822|         5127138382|         5

In [26]:
df.sample(0.001).show(10)

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+
|agg_trade_id|    price|quantity|first_trade_id|last_trade_id|       timestamp|is_buyer_maker|is_best_match|
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+
|  3640494378|115730.52|  8.0E-5|    5122978118|   5122978118|1754006414992420|          true|         true|
|  3640494397|115730.52|     0.2|    5122978176|   5122978176|1754006417095514|         false|         true|
|  3640498085|115809.36|  1.5E-4|    5122985673|   5122985675|1754006711532358|         false|         true|
|  3640498760| 115788.0|  3.0E-4|    5122987402|   5122987402|1754006751612978|          true|         true|
|  3640499656| 115740.7|  5.0E-5|    5122989371|   5122989371|1754006855380139|         false|         true|
|  3640499890|115777.29|  3.0E-4|    5122989913|   5122989918|1754006869316143|          true|         true|
|  3640501343| 1157

In [27]:
df.withColumn("ingest_date", F.current_date()).withColumn("ingest_timestamp", F.current_timestamp()).show(truncate=False)

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------------+
|agg_trade_id|price    |quantity|first_trade_id|last_trade_id|timestamp       |is_buyer_maker|is_best_match|ingest_date|ingest_timestamp          |
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------------+
|3640494121  |115764.07|0.22677 |5122977554    |5122977554   |1754006400328945|true          |true         |2025-10-03 |2025-10-03 02:28:16.997391|
|3640494122  |115764.08|0.00145 |5122977555    |5122977555   |1754006400345714|false         |true         |2025-10-03 |2025-10-03 02:28:16.997391|
|3640494123  |115764.08|2.1E-4  |5122977556    |5122977556   |1754006400350235|false         |true         |2025-10-03 |2025-10-03 02:28:16.997391|
|3640494124  |115764.08|4.1E-4  |5122977557    |5122977557   |1754006400492405|false         |true         |2025

In [28]:
df = df.withColumn("ingest_date", F.current_date()) \
    .withColumn("ingest_timestamp", F.current_timestamp())

In [29]:
output_path = f"s3a://{bucket}/landing_zone/spot/daily/aggTrades/BTCUSDT/2025_08_01"
df.write.mode("overwrite").parquet(output_path)
print(f"Parquet written to: {output_path}")

25/10/03 02:28:43 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Parquet written to: s3a://crypto-data-lake/landing_zone/spot/daily/aggTrades/BTCUSDT/2025_08_01


In [4]:
output_path = f"s3a://{bucket}/landing_zone/spot/daily/aggTrades/BTCUSDT/2025_08_01"
df = spark.read.parquet(output_path)

25/10/03 15:49:30 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [5]:
df.printSchema()

root
 |-- agg_trade_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- first_trade_id: long (nullable = true)
 |-- last_trade_id: long (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)
 |-- ingest_date: date (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)



In [6]:
df.describe().show()

25/10/03 15:49:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-------+-----------------+------------------+--------------------+-------------------+-------------------+--------------------+
|summary|     agg_trade_id|             price|            quantity|     first_trade_id|      last_trade_id|           timestamp|
+-------+-----------------+------------------+--------------------+-------------------+-------------------+--------------------+
|  count|          1314072|           1314072|             1314072|            1314072|            1314072|             1314072|
|   mean|   3.6411511565E9|114772.51094437456|0.018634520832961927|5.124889106730406E9|5.124889108896769E9|1.754049467280027...|
| stddev|379340.0558048148| 815.0145912458705| 0.13296616458618138| 1227424.3654440136| 1227424.8011467636|2.531215074523887...|
|    min|       3640494121|         112722.58|              1.0E-5|         5122977554|         5122977554|    1754006400328945|
|    max|       3641808192|          116052.0|             26.5822|         5127138382|         5

In [7]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+------------+-----+--------+--------------+-------------+---------+--------------+-------------+-----------+----------------+
|agg_trade_id|price|quantity|first_trade_id|last_trade_id|timestamp|is_buyer_maker|is_best_match|ingest_date|ingest_timestamp|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+-----------+----------------+
|           0|    0|       0|             0|            0|        0|             0|            0|          0|               0|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+-----------+----------------+



In [8]:
df.sample(0.001).show(truncate=False)

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+-------------------------+
|agg_trade_id|price    |quantity|first_trade_id|last_trade_id|timestamp       |is_buyer_maker|is_best_match|ingest_date|ingest_timestamp         |
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+-------------------------+
|3640494363  |115730.52|0.011   |5122978102    |5122978102   |1754006414345867|true          |true         |2025-10-03 |2025-10-03 15:47:39.76051|
|3640494517  |115729.74|0.0023  |5122978366    |5122978366   |1754006433847809|true          |true         |2025-10-03 |2025-10-03 15:47:39.76051|
|3640494643  |115704.79|0.00113 |5122978602    |5122978602   |1754006441033728|false         |true         |2025-10-03 |2025-10-03 15:47:39.76051|
|3640496366  |115720.56|0.00133 |5122981992    |5122981993   |1754006572022774|true          |true         |2025-10-03

In [1]:
!jupyter nbconvert --to script landing_job.ipynb

[NbConvertApp] Converting notebook landing_job.ipynb to script
[NbConvertApp] Writing 4346 bytes to landing_job.py
